# 06 — Business aggregation

Agrégation des résultats ABSA pour Moisturizers, Treatments et Cleansers. Les taux utilisent `sample_weight` pour corriger le sur-échantillonnage des ratings.

In [1]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE

while not (ROOT / "config" / "project_config.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

assert (ROOT / "config" / "project_config.json").exists(), "Project root not found."

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from IPython.display import display

CONFIG = json.loads(
    (ROOT / "config" / "project_config.json").read_text(encoding="utf-8")
)

SUPPORT = CONFIG.get("support_threshold_business", 10)

OUT_DIR = ROOT / "data" / "processed" / "sample_2000_business"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Business output:", OUT_DIR)
print("Support threshold:", SUPPORT)


ROOT: C:\Users\wiame.bourass\Downloads\sephora-consumer-voice-intelligence-llm-first\sephora-consumer-voice-intelligence-llm-first
Business output: C:\Users\wiame.bourass\Downloads\sephora-consumer-voice-intelligence-llm-first\sephora-consumer-voice-intelligence-llm-first\data\processed\sample_2000_business
Support threshold: 30


## 1. Chargement des résultats

In [2]:
RUN_DIR = (
    ROOT
    / "data"
    / "interim"
    / "test_2000_gpt_oss_20b_batch4_chunks200_v2"
)

PAIRS_PATH = RUN_DIR / "pairs.parquet"
SAMPLE_PATH = RUN_DIR / "sample_2000.parquet"
STATUS_PATH = RUN_DIR / "status.parquet"
REQUESTS_PATH = RUN_DIR / "requests.csv"

assert PAIRS_PATH.exists(), f"Introuvable : {PAIRS_PATH}"
assert SAMPLE_PATH.exists(), f"Introuvable : {SAMPLE_PATH}"

pairs_raw = pd.read_parquet(PAIRS_PATH)
sample = pd.read_parquet(SAMPLE_PATH)

print("Segments dans l'échantillon :", f"{len(sample):,}")
print("Reviews distinctes :", f"{sample['review_id'].nunique():,}")
print("Paires ABSA :", f"{len(pairs_raw):,}")
print("Segments avec au moins un aspect :", f"{pairs_raw['segment_id'].nunique():,}")

display(pairs_raw.head())


Segments dans l'échantillon : 2,000
Reviews distinctes : 1,995
Paires ABSA : 1,449
Segments avec au moins un aspect : 1,194


,review_id,author_id,product_id,product_name_catalog,brand_name_catalog,skin_type,rating,is_recommended,submission_time,primary_category,...,_rating_group,_stratum,sample_weight,aspect_id,sentiment,evidence,evidence_exact_match,run_name,batch_id,prompt_version
0,reviews_500-750::5007,1325857542,P461949,The Concentrate Serum,La Mer,dry,5,1.0,2021-11-05,Skincare,...,positive_4_5,Treatments | positive_4_5,816.867788,efficacy_results,positive,help my redness and dry skin,True,gpt_oss_20b_test_2000_chunk_00,0,absa_v2_llm_first
1,reviews_500-750::5007,1325857542,P461949,The Concentrate Serum,La Mer,dry,5,1.0,2021-11-05,Skincare,...,positive_4_5,Treatments | positive_4_5,816.867788,hydration_dryness,positive,help my redness and dry skin,True,gpt_oss_20b_test_2000_chunk_00,0,absa_v2_llm_first
2,reviews_0-250::206616,35746150235,P458219,Watermelon Glow PHA + BHA Pore-Tight Toner,Glow Recipe,combination,3,0.0,2022-03-11,Skincare,...,neutral_3,Cleansers | neutral_3,450.537037,texture_finish,negative,pretty goopy instead of a smooth liquid,True,gpt_oss_20b_test_2000_chunk_00,0,absa_v2_llm_first
3,reviews_0-250::206616,35746150235,P458219,Watermelon Glow PHA + BHA Pore-Tight Toner,Glow Recipe,combination,3,0.0,2022-03-11,Skincare,...,neutral_3,Cleansers | neutral_3,450.537037,efficacy_results,positive,make my face glow,True,gpt_oss_20b_test_2000_chunk_00,0,absa_v2_llm_first
4,reviews_1250-end::33437,10655985690,P469544,C-50 Blemish Night Treatment,The INKEY List,normal,1,0.0,2022-01-05,Skincare,...,negative_1_2,Treatments | negative_1_2,252.968750,price_value,negative,As cheap as this is,True,gpt_oss_20b_test_2000_chunk_00,0,absa_v2_llm_first


## 2. Métadonnées et poids

In [3]:
meta_candidates = [
    "review_id",
    "author_id",
    "product_id",
    "product_name_catalog",
    "brand_name_catalog",
    "skin_type",
    "rating",
    "is_recommended",
    "submission_time",
    "primary_category",
    "secondary_category",
    "tertiary_category",
    "_rating_group",
    "_stratum",
    "sample_weight",
]

meta_cols = [
    c for c in meta_candidates
    if c in sample.columns and c not in pairs_raw.columns
]

pairs = pairs_raw.merge(
    sample[["segment_id"] + meta_cols].drop_duplicates("segment_id"),
    on="segment_id",
    how="left",
    validate="many_to_one",
)

if "sample_weight" not in pairs.columns:
    pairs["sample_weight"] = 1.0
    print("⚠️ sample_weight absent : poids=1 utilisé.")
else:
    assert pairs["sample_weight"].notna().all(), "Certains poids sont manquants."

print("Shape pairs enrichi :", pairs.shape)
print("Colonnes :", pairs.columns.tolist())

display(
    pairs[
        [c for c in [
            "segment_id",
            "secondary_category",
            "rating",
            "aspect_id",
            "sentiment",
            "sample_weight",
        ] if c in pairs.columns]
    ].head(10)
)


Shape pairs enrichi : (1449, 29)
Colonnes : ['review_id', 'author_id', 'product_id', 'product_name_catalog', 'brand_name_catalog', 'skin_type', 'rating', 'is_recommended', 'submission_time', 'primary_category', 'secondary_category', 'tertiary_category', 'review_text', 'segment_text', 'segment_index', 'segment_id', 'segment_word_count', 'has_contrast_marker', 'is_short_segment', '_rating_group', '_stratum', 'sample_weight', 'aspect_id', 'sentiment', 'evidence', 'evidence_exact_match', 'run_name', 'batch_id', 'prompt_version']


,segment_id,secondary_category,rating,aspect_id,sentiment,sample_weight
0,reviews_500-750::5007::s3,Treatments,5,efficacy_results,positive,816.867788
1,reviews_500-750::5007::s3,Treatments,5,hydration_dryness,positive,816.867788
2,reviews_0-250::206616::s0,Cleansers,3,texture_finish,negative,450.537037
3,reviews_0-250::206616::s0,Cleansers,3,efficacy_results,positive,450.537037
4,reviews_1250-end::33437::s0,Treatments,1,price_value,negative,252.968750
5,reviews_0-250::572051::s3,Cleansers,5,packaging,positive,817.919540
6,reviews_250-500::143172::s2,Moisturizers,4,efficacy_results,positive,811.936567
7,reviews_750-1250::104089::s0,Treatments,5,efficacy_results,positive,816.867788
8,reviews_250-500::86731::s1,Moisturizers,5,efficacy_results,positive,811.936567
9,reviews_0-250::60159::s2,Cleansers,5,irritation_sensitivity,negative,817.919540


## 3. Contrôles

In [4]:
print("Catégories présentes :")
display(
    sample["secondary_category"]
    .value_counts(dropna=False)
    .to_frame("n_segments")
)

print("\nRating groups :")
if "_rating_group" in sample.columns:
    display(
        sample["_rating_group"]
        .value_counts(dropna=False)
        .to_frame("n_segments")
    )

print("\nDiversité :")
print("Marques :", sample["brand_name_catalog"].nunique(dropna=True))
print("Produits :", sample["product_id"].nunique(dropna=True))
print("Reviews :", sample["review_id"].nunique(dropna=True))

assert set(sample["secondary_category"].dropna().unique()).issubset(
    {"Moisturizers", "Treatments", "Cleansers"}
), "Le sample contient des catégories hors périmètre."


Catégories présentes :


,n_segments
secondary_category,
Moisturizers,824
Treatments,640
Cleansers,536



Rating groups :


,n_segments
_rating_group,
positive_4_5,1300
negative_1_2,500
neutral_3,200



Diversité :
Marques : 99
Produits : 645
Reviews : 1995


## 4. Agrégation pondérée

`n_mentions` reste un volume observé ; les taux de sentiment sont pondérés.

In [5]:
SENTIMENTS = ["positive", "neutral", "negative"]

def aggregate_weighted_aspect_sentiment(df, group_cols=None, support_threshold=SUPPORT):
    group_cols = list(group_cols or [])

    required = {"aspect_id", "sentiment", "sample_weight"}
    missing = required - set(df.columns)
    assert not missing, f"Colonnes manquantes : {missing}"

    work = df.copy()
    work = work[work["sentiment"].isin(SENTIMENTS)].copy()

    keys = group_cols + ["aspect_id"]

    # Nombre brut de mentions observées
    raw = (
        work.groupby(keys, dropna=False)
        .size()
        .rename("n_mentions")
    )

    # Somme des poids par groupe/aspect
    weighted_total = (
        work.groupby(keys, dropna=False)["sample_weight"]
        .sum()
        .rename("weighted_mentions")
    )

    # Comptages pondérés par sentiment
    weighted_sent = (
        work.groupby(keys + ["sentiment"], dropna=False)["sample_weight"]
        .sum()
        .unstack("sentiment", fill_value=0.0)
    )

    for sentiment in SENTIMENTS:
        if sentiment not in weighted_sent.columns:
            weighted_sent[sentiment] = 0.0

    result = pd.concat(
        [raw, weighted_total, weighted_sent[SENTIMENTS]],
        axis=1,
    ).reset_index()

    for sentiment in SENTIMENTS:
        result[f"{sentiment}_rate"] = (
            result[sentiment]
            / result["weighted_mentions"].replace(0, np.nan)
        )

    result["net_sentiment"] = (
        result["positive_rate"] - result["negative_rate"]
    )

    result["meets_support"] = (
        result["n_mentions"] >= support_threshold
    )

    return result.sort_values(
        ["n_mentions"],
        ascending=False,
    ).reset_index(drop=True)


## 5. Vue globale par aspect

In [6]:
overall = aggregate_weighted_aspect_sentiment(
    pairs,
    group_cols=[],
)

display(overall)

overall.to_parquet(
    OUT_DIR / "agg_overall_aspect.parquet",
    index=False,
)


,aspect_id,n_mentions,weighted_mentions,positive,neutral,negative,positive_rate,neutral_rate,negative_rate,net_sentiment,meets_support
0,efficacy_results,419,284702.448571,252598.108463,2223.435924,29880.904184,0.887235,0.007810,0.104955,0.782281,True
1,texture_finish,270,180110.568159,135170.961045,8664.686515,36274.920599,0.750489,0.048108,0.201404,0.549085,True
2,hydration_dryness,195,129958.821828,102436.437059,3602.258068,23920.126701,0.788222,0.027718,0.184059,0.604163,True
3,acne_breakouts,130,68110.374385,38013.628724,1312.770286,28783.975374,0.558118,0.019274,0.422608,0.135510,True
4,fragrance_smell,129,82129.688697,53948.273291,8922.343190,19259.072216,0.656867,0.108637,0.234496,0.422371,True
5,irritation_sensitivity,108,66868.560938,30781.933208,1069.836538,35016.791191,0.460335,0.015999,0.523666,-0.063331,True
6,price_value,84,51294.521435,23136.345438,5013.495089,23144.680908,0.451049,0.097739,0.451212,-0.000163,True
7,application_absorption,66,43207.981634,35939.554395,1628.804356,5639.622884,0.831780,0.037697,0.130523,0.701258,True
8,packaging,48,32062.370157,20332.234562,3427.952691,8302.182904,0.634146,0.106915,0.258939,0.375208,True


## 6. Comparaison par catégorie

In [7]:
category_agg = aggregate_weighted_aspect_sentiment(
    pairs,
    group_cols=["secondary_category"],
)

category_agg.to_parquet(
    OUT_DIR / "agg_category_aspect.parquet",
    index=False,
)

display(
    category_agg.sort_values(
        ["secondary_category", "n_mentions"],
        ascending=[True, False],
    )
)


,secondary_category,aspect_id,n_mentions,weighted_mentions,positive,neutral,negative,positive_rate,neutral_rate,negative_rate,net_sentiment,meets_support
3,Cleansers,efficacy_results,112,75053.750119,68151.807667,450.537037,6451.405415,0.908040,0.006003,0.085957,0.822083,True
5,Cleansers,texture_finish,72,47393.734365,35178.051343,2701.183994,9514.499028,0.742251,0.056995,0.200754,0.541497,True
8,Cleansers,hydration_dryness,53,33932.128105,24748.208984,3602.258068,5581.661053,0.729344,0.106161,0.164495,0.564850,True
10,Cleansers,fragrance_smell,46,29347.679657,20455.499619,3354.832695,5537.347344,0.697006,0.114313,0.188681,0.508325,True
15,Cleansers,acne_breakouts,36,19176.208441,13581.563390,1312.770286,4281.874764,0.708251,0.068458,0.223291,0.484960,True
16,Cleansers,irritation_sensitivity,33,21528.355645,13987.786718,0.000000,7540.568927,0.649738,0.000000,0.350262,0.299476,True
19,Cleansers,price_value,27,15441.009245,7856.126608,2086.376117,5498.506519,0.508783,0.135119,0.356098,0.152686,False
25,Cleansers,application_absorption,11,6918.249938,6175.973819,0.000000,742.276119,0.892708,0.000000,0.107292,0.785415,False
26,Cleansers,packaging,9,5079.299194,2701.183994,1312.770286,1065.344913,0.531802,0.258455,0.209743,0.322060,False
1,Moisturizers,efficacy_results,146,102932.452057,92860.650556,250.359223,9821.442278,0.902151,0.002432,0.095416,0.806735,True


## 7. Par marque

Interpréter surtout les lignes avec un support suffisant.

In [8]:
brand_agg = aggregate_weighted_aspect_sentiment(
    pairs,
    group_cols=["brand_name_catalog"],
)

brand_agg.to_parquet(
    OUT_DIR / "agg_brand_aspect.parquet",
    index=False,
)

display(
    brand_agg[
        brand_agg["meets_support"]
    ].head(30)
)


,brand_name_catalog,aspect_id,n_mentions,weighted_mentions,positive,neutral,negative,positive_rate,neutral_rate,negative_rate,net_sentiment,meets_support


## 8. Par produit

In [9]:
product_group_cols = [
    c for c in [
        "product_id",
        "product_name_catalog",
        "brand_name_catalog",
    ]
    if c in pairs.columns
]

product_agg = aggregate_weighted_aspect_sentiment(
    pairs,
    group_cols=product_group_cols,
)

product_agg.to_parquet(
    OUT_DIR / "agg_product_aspect.parquet",
    index=False,
)

display(
    product_agg[
        product_agg["meets_support"]
    ].head(30)
)


,product_id,product_name_catalog,brand_name_catalog,aspect_id,n_mentions,weighted_mentions,positive,neutral,negative,positive_rate,neutral_rate,negative_rate,net_sentiment,meets_support


## 9. Par type de peau

Le type de peau est auto-déclaré ; les résultats sont descriptifs.

In [10]:
skin_source = pairs[
    pairs["skin_type"].notna()
].copy()

skin_agg = aggregate_weighted_aspect_sentiment(
    skin_source,
    group_cols=["skin_type"],
)

skin_agg.to_parquet(
    OUT_DIR / "agg_skin_type_aspect.parquet",
    index=False,
)

display(
    skin_agg.sort_values(
        ["aspect_id", "n_mentions"],
        ascending=[True, False],
    ).head(50)
)


,skin_type,aspect_id,n_mentions,weighted_mentions,positive,neutral,negative,positive_rate,neutral_rate,negative_rate,net_sentiment,meets_support
4,combination,acne_breakouts,68,32475.527743,17625.797487,817.919540,14031.810716,0.542741,0.025186,0.432073,0.110668,True
16,dry,acne_breakouts,23,13010.535645,8973.907264,0.000000,4036.628381,0.689742,0.000000,0.310258,0.379483,False
19,oily,acne_breakouts,19,11207.220008,4894.499543,247.425373,6065.295092,0.436727,0.022077,0.541195,-0.104468,False
30,normal,acne_breakouts,9,5065.841900,3253.729242,247.425373,1564.687285,0.642288,0.048842,0.308870,0.333418,False
12,combination,application_absorption,38,25268.757910,20065.412078,811.936567,4391.409264,0.794080,0.032132,0.173788,0.620292,True
29,normal,application_absorption,9,5866.696243,4802.403081,816.867788,247.425373,0.818587,0.139238,0.042175,0.776413,False
32,dry,application_absorption,7,3998.499615,3498.105492,0.000000,500.394123,0.874855,0.000000,0.125145,0.749709,False
34,oily,application_absorption,4,2693.709673,2440.740923,0.000000,252.968750,0.906089,0.000000,0.093911,0.812178,False
0,combination,efficacy_results,210,142469.164572,125836.718361,703.062348,15929.383863,0.883256,0.004935,0.111809,0.771446,True
5,dry,efficacy_results,62,43659.488297,39237.506639,252.968750,4169.012908,0.898717,0.005794,0.095489,0.803227,True


## 10. Catégorie × aspect × sentiment

In [11]:
category_sentiment = (
    pairs.groupby(
        ["secondary_category", "aspect_id", "sentiment"],
        dropna=False,
    )
    .agg(
        n_mentions=("segment_id", "size"),
        weighted_mentions=("sample_weight", "sum"),
    )
    .reset_index()
)

category_sentiment.to_parquet(
    OUT_DIR / "category_aspect_sentiment_long.parquet",
    index=False,
)

display(category_sentiment.head(30))


,secondary_category,aspect_id,sentiment,n_mentions,weighted_mentions
0,Cleansers,acne_breakouts,negative,15,4281.874764
1,Cleansers,acne_breakouts,neutral,3,1312.770286
2,Cleansers,acne_breakouts,positive,18,13581.563390
3,Cleansers,application_absorption,negative,3,742.276119
4,Cleansers,application_absorption,positive,8,6175.973819
5,Cleansers,efficacy_results,negative,19,6451.405415
6,Cleansers,efficacy_results,neutral,1,450.537037
7,Cleansers,efficacy_results,positive,92,68151.807667
8,Cleansers,fragrance_smell,negative,13,5537.347344
9,Cleansers,fragrance_smell,neutral,5,3354.832695


## Sorties

Résultats enregistrés dans `data/processed/sample_2000_business/`.